# Azure 函数调用示例

> 注意：openai 库已有更新的版本。参见 https://github.com/openai/openai-python/discussions/742

本 notebook 展示如何在 Azure OpenAI 服务中使用函数调用（function calling）功能。函数允许 chat completions 的调用方定义模型可以使用的能力，从而将模型的
功能扩展到外部工具和数据源。

你可以在 OpenAI 的博客上进一步了解聊天函数：https://openai.com/blog/function-calling-and-other-api-updates

**注意**：聊天函数要求使用 gpt-4 和 gpt-35-turbo 中以 `-0613` 标签开始的模型版本。旧版本的模型不支持该功能。

## 设置

首先，安装所需的依赖项。

In [ ]:
! pip install "openai>=0.28.1,<1.0.0"
# (Optional) If you want to use Microsoft Active Directory
! pip install azure-identity

In [3]:
import os
import openai


此外，要正确访问 Azure OpenAI 服务，我们需要在 [Azure 门户](https://portal.azure.com) 中创建相应的资源（关于具体操作，可参阅 [Microsoft 文档](https://learn.microsoft.com/en-us/azure/cognitive-services/openai/how-to/create-resource?pivots=web-portal) 中的详细指南）

资源创建完成后，首先需要用到的是它的终结点（endpoint）。你可以在 *"资源管理"* 部分下的 *"密钥和终结点"* 中找到终结点。拿到它之后，我们用这些信息来设置 SDK：

In [27]:
openai.api_base = "" # Add your endpoint here

# functions is only supported by the 2023-07-01-preview API version
openai.api_version = "2023-07-01-preview"

### 身份验证

Azure OpenAI 服务支持多种身份验证机制，包括 API 密钥和 Azure 凭据。

In [4]:
use_azure_active_directory = False


#### 使用 API 密钥进行身份验证

要将 OpenAI SDK 设置为使用 *Azure API 密钥*，需要将 `api_type` 设置为 `azure`，并将 `api_key` 设置为与你的终结点关联的密钥（你可以在 [Azure 门户](https://portal.azure.com) 的 *"资源管理"* 下的 *"密钥和终结点"* 中找到该密钥）

In [ ]:
if not use_azure_active_directory:
    openai.api_type = "azure"
    openai.api_key = os.environ["OPENAI_API_KEY"]

> 注意：在本示例中，我们通过在代码中设置变量来让库使用 Azure API。在开发时，建议改为设置环境变量：

```
OPENAI_API_BASE
OPENAI_API_KEY
OPENAI_API_TYPE
OPENAI_API_VERSION
```

#### 使用 Microsoft Active Directory 进行身份验证
现在来看看如何通过 Microsoft Active Directory 身份验证获取密钥。

In [ ]:
from azure.identity import DefaultAzureCredential

if use_azure_active_directory:
    default_credential = DefaultAzureCredential()
    token = default_credential.get_token("https://cognitiveservices.azure.com/.default")

    openai.api_type = "azure_ad"
    openai.api_key = token.token

令牌在一段时间内有效，过期后即失效。为确保每个请求都携带有效的令牌，你可以通过挂接 requests.auth 来刷新即将过期的令牌：

In [ ]:
import typing
import time
import requests

if typing.TYPE_CHECKING:
    from azure.core.credentials import TokenCredential

class TokenRefresh(requests.auth.AuthBase):

    def __init__(self, credential: "TokenCredential", scopes: typing.List[str]) -> None:
        self.credential = credential
        self.scopes = scopes
        self.cached_token: typing.Optional[str] = None

    def __call__(self, req):
        if not self.cached_token or self.cached_token.expires_on - time.time() < 300:
            self.cached_token = self.credential.get_token(*self.scopes)
        req.headers["Authorization"] = f"Bearer {self.cached_token.token}"
        return req

if use_azure_active_directory:
    session = requests.Session()
    session.auth = TokenRefresh(default_credential, ["https://cognitiveservices.azure.com/.default"])

    openai.requestssession = session

## 函数

完成设置和身份验证后，你就可以在 Azure OpenAI 服务中使用函数了。这将分为以下几个步骤：

1. 定义函数
2. 将函数定义传入 chat completions API
3. 使用响应中的参数调用函数
4. 将函数的响应回传给 chat completions API

#### 1. 定义函数

可以定义一个函数列表，其中每个函数都包含函数名、可选的描述，以及函数接受的参数（以 JSON schema 描述）。

In [21]:
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "format": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The temperature unit to use. Infer this from the users location.",
                },
            },
            "required": ["location"],
        },
    }
]

#### 2. 将函数定义传入 chat completions API

现在我们可以将函数传入 chat completions API。如果模型判断应当调用该函数，choice 中的 `finish_reason` 会被设置为 "function_call"，并且 `message` 中会包含要调用哪个函数及其参数的详细信息。你也可以选择设置 `function_call` 关键字参数，强制模型调用某个特定函数（例如 `function_call={"name": get_current_weather}`）。默认情况下它被设置为 `auto`，由模型自行决定是否调用函数。 

In [ ]:
messages = [
    {"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."},
    {"role": "user", "content": "What's the weather like today in Seattle?"}
]

chat_completion = openai.ChatCompletion.create(
    deployment_id="gpt-35-turbo-0613",
    messages=messages,
    functions=functions,
)
print(chat_completion)

{
  "choices": [
    {
      "content_filter_results": {},
      "finish_reason": "function_call",
      "index": 0,
      "message": {
        "function_call": {
          "arguments": "{\n  \"location\": \"Seattle, WA\"\n}",
          "name": "get_current_weather"
        },
        "role": "assistant"
      }
    }
  ],
  "created": 1689702512,
  "id": "chatcmpl-7dj6GkYdM7Vw9eGn02bc2qqjN70Ps",
  "model": "gpt-4",
  "object": "chat.completion",
  "prompt_annotations": [
    {
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        }
      },
      "prompt_index": 0
    }
  ],
  "usage": {
    "completion_tokens": 18,
    "prompt_tokens": 115,
    "total_to

#### 3. 使用响应中的参数调用函数

函数调用的名称将是最初提供的函数之一，参数则是符合函数定义中 schema 的 JSON。

In [ ]:
import json

def get_current_weather(request):
    """
    This function is for illustrative purposes.
    The location and unit should be used to determine weather
    instead of returning a hardcoded response.
    """
    location = request.get("location")
    unit = request.get("unit")
    return {"temperature": "22", "unit": "celsius", "description": "Sunny"}

function_call =  chat_completion.choices[0].message.function_call
print(function_call.name)
print(function_call.arguments)

if function_call.name == "get_current_weather":
    response = get_current_weather(json.loads(function_call.arguments))

get_current_weather
{
  "location": "Seattle, WA"
}


#### 4. 将函数的响应回传给 chat completions API

函数的响应应序列化为一条 role 设置为 "function" 的新消息。随后模型将使用这些响应数据来组织它的回答。

In [ ]:
messages.append(
    {
        "role": "function",
        "name": "get_current_weather",
        "content": json.dumps(response)
    }
)

function_completion = openai.ChatCompletion.create(
    deployment_id="gpt-35-turbo-0613",
    messages=messages,
    functions=functions,
)

print(function_completion.choices[0].message.content.strip())

Today in Seattle, the weather is sunny with a temperature of 22 degrees celsius.
